# Exploration des données météo (OpenWeatherMap)

**Objectif :** comprendre les données avant de construire la suite du pipeline. Le notebook répond à cinq questions :

1. Quelle est la structure d'une réponse de l'API, et que signifie chaque champ ?
2. Quelles sont les unités, et comment convertir les dates ?
3. Quels champs peuvent être absents, et pourquoi ?
4. Les valeurs sont-elles plausibles et cohérentes (plages, doublons, villes) ?
5. Quelles colonnes garder, et sous quelle forme, dans le futur modèle dbt `stg_weather` ?

**Limite à garder en tête :** 2 ou 3 extractions donnent 10 à 15 lignes. C'est suffisant pour comprendre la structure et la qualité des données, mais pas pour tirer des conclusions statistiques sur la météo.

## 0. Produire les fichiers

Pas besoin de Google Cloud pour cette exploration : la clé suffit. Dans `.env`, renseigner `OPENWEATHER_API_KEY=ta_cle`, puis lancer l'extraction 2 ou 3 fois, idéalement à quelques heures d'intervalle (matin, midi, soir), dans des dossiers différents :

```bash
python -m pipeline_meteo.extract --output-dir data/exploration/run1
# quelques heures plus tard
python -m pipeline_meteo.extract --output-dir data/exploration/run2
python -m pipeline_meteo.extract --output-dir data/exploration/run3
```

Le dossier `data/` est exclu de Git : les fichiers restent sur ton poste.

Pour ouvrir ce notebook : `pip install -r requirements-dev.txt`, puis ouvrir le fichier dans VS Code et choisir le noyau Python du `.venv`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Le notebook fonctionne qu'il soit lancé depuis la racine du projet ou depuis notebooks/
ROOT = Path.cwd() if (Path.cwd() / "pipeline_meteo").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))  # pour importer le code du projet (pipeline_meteo)
DATA_DIR = ROOT / "data" / "exploration"

files = sorted(DATA_DIR.rglob("*.ndjson"))
print(f"{len(files)} fichier(s) trouvé(s) :")
for f in files:
    print(" -", f.relative_to(ROOT))
assert files, "Aucun fichier : lancer d'abord l'extraction (section 0)."

## 1. Charger les fichiers et regarder une réponse brute

In [ ]:
records = []
for f in files:
    with open(f, encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                rec = json.loads(line)
                rec["source_file"] = str(f.relative_to(DATA_DIR))
                records.append(rec)

print(f"{len(records)} lignes au total")
print(json.dumps(records[0]["payload"], indent=2, ensure_ascii=False))

**Lecture d'une réponse (API « Current weather », unités `metric`) :**

| Champ | Signification | Unité |
|---|---|---|
| `coord.lat`, `coord.lon` | Position utilisée par l'API | degrés |
| `weather[0].main`, `.description` | Type de temps (Clear, Clouds, Rain…) et libellé détaillé | texte |
| `main.temp`, `main.feels_like` | Température et ressentie | °C |
| `main.temp_min`, `main.temp_max` | Min et max observés **à cet instant** sur la zone (pas sur la journée) | °C |
| `main.pressure` | Pression au niveau de la mer | hPa |
| `main.humidity` | Humidité relative | % |
| `visibility` | Visibilité, plafonnée à 10 000 | m |
| `wind.speed`, `wind.deg`, `wind.gust` | Vitesse, direction (0° = nord), rafales | m/s, degrés |
| `clouds.all` | Couverture nuageuse | % |
| `rain.1h`, `snow.1h` | Précipitations de la dernière heure, **seulement s'il y en a** | mm |
| `dt` | Instant de l'observation | timestamp Unix UTC |
| `sys.sunrise`, `sys.sunset` | Lever et coucher du soleil | timestamp Unix UTC |
| `timezone` | Décalage horaire local | secondes |
| `id`, `name` | Identifiant et nom de la ville côté API | |

À vérifier sur tes données : ce tableau vient de la documentation de l'API, et la section 3 montre les champs réellement présents.

## 2. Aplatir le JSON en tableau

In [ ]:
df = pd.json_normalize(records, sep=".")

# `weather` est une liste : on regarde combien d'éléments elle contient avant de garder le premier
n_weather = df["payload.weather"].apply(len)
print("Nombre d'éléments dans weather :", n_weather.value_counts().to_dict())

weather0 = pd.json_normalize(df["payload.weather"].apply(lambda w: w[0] if w else {}))
weather0.columns = [f"payload.weather0.{c}" for c in weather0.columns]
df = pd.concat([df.drop(columns="payload.weather"), weather0], axis=1)

# Noms plus courts pour la suite
df.columns = [c.replace("payload.", "") for c in df.columns]
print(df.shape)
df.head()

## 3. Types, champs présents et valeurs manquantes

In [ ]:
overview = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "non_nuls": df.notna().sum(),
    "pct_manquant": (df.isna().mean() * 100).round(1),
    "exemple": df.apply(lambda s: s.dropna().iloc[0] if s.notna().any() else None),
}).sort_values("pct_manquant", ascending=False)
overview

**Comment lire les champs manquants.** Certains champs n'existent que dans certaines conditions :

- `rain.1h` et `snow.1h` n'apparaissent que s'il a plu ou neigé pendant la dernière heure. Leur absence veut dire « pas de précipitation mesurée », et on pourra la remplacer par 0 dans dbt.
- `wind.gust` n'apparaît que s'il y a des rafales.
- `sea_level` et `grnd_level` peuvent manquer selon la station.

Un champ manquant n'est donc pas forcément une erreur. Il faut regarder la règle de l'API avant de décider comment le traiter.

## 4. Dates et fraîcheur des observations

In [ ]:
df["observed_at_utc"] = pd.to_datetime(df["dt"], unit="s", utc=True)
df["extracted_at"] = pd.to_datetime(df["extracted_at"], utc=True)
df["observed_at_local"] = df["observed_at_utc"] + pd.to_timedelta(df["timezone"], unit="s")
df["sunrise_utc"] = pd.to_datetime(df["sys.sunrise"], unit="s", utc=True)
df["sunset_utc"] = pd.to_datetime(df["sys.sunset"], unit="s", utc=True)
df["is_day"] = df["observed_at_utc"].between(df["sunrise_utc"], df["sunset_utc"])

# Fraîcheur : combien de temps entre l'observation et notre appel à l'API ?
df["age_minutes"] = (df["extracted_at"] - df["observed_at_utc"]).dt.total_seconds() / 60

df[["source_file", "city_name", "observed_at_utc", "observed_at_local",
    "extracted_at", "age_minutes", "is_day"]].sort_values(["city_name", "observed_at_utc"])

**À retenir :** `dt` est l'heure de **l'observation**, et non celle de notre appel. L'écart (`age_minutes`) montre la fraîcheur réelle de la donnée. Dans dbt, on gardera les deux horodatages : `observed_at` pour l'analyse, `extracted_at` pour la traçabilité.

## 5. Contrôles de qualité

In [ ]:
checks = {
    "code HTTP de la réponse = 200": (df["cod"] == 200).all(),
    "température entre -50 et 50 °C": df["main.temp"].between(-50, 50).all(),
    "humidité entre 0 et 100 %": df["main.humidity"].between(0, 100).all(),
    "pression entre 870 et 1085 hPa": df["main.pressure"].between(870, 1085).all(),
    "vent >= 0": (df["wind.speed"] >= 0).all(),
    "direction du vent entre 0 et 360°": df["wind.deg"].between(0, 360).all(),
    "nuages entre 0 et 100 %": df["clouds.all"].between(0, 100).all(),
    "temp_min <= temp <= temp_max": ((df["main.temp_min"] <= df["main.temp"])
                                     & (df["main.temp"] <= df["main.temp_max"])).all(),
}
pd.Series(checks, name="OK").to_frame()

In [ ]:
# Doublons d'observation : deux extractions rapprochées peuvent renvoyer la même mesure (même ville, même dt).
dup = df.duplicated(subset=["id", "dt"], keep=False)
print(f"Lignes partageant la même observation (id, dt) : {dup.sum()}")
df.loc[dup, ["source_file", "city_name", "id", "observed_at_utc"]]

In [ ]:
# Ville demandée et ville renvoyée par l'API : le nom et la position peuvent différer.
from pipeline_meteo.extract import load_cities

requested = pd.DataFrame(load_cities()).rename(columns={"name": "city_name", "lat": "lat_req", "lon": "lon_req"})
cmp = (df[["city_name", "name", "id", "coord.lat", "coord.lon"]].drop_duplicates()
       .merge(requested[["city_name", "lat_req", "lon_req"]], on="city_name"))
# Distance approchée (1° de latitude ≈ 111 km ; 1° de longitude ≈ 111 km × cos(latitude))
dlat_km = (cmp["coord.lat"] - cmp["lat_req"]) * 111
dlon_km = (cmp["coord.lon"] - cmp["lon_req"]) * 111 * np.cos(np.radians(cmp["lat_req"]))
cmp["ecart_km_approx"] = np.sqrt(dlat_km ** 2 + dlon_km ** 2).round(1)
cmp

**À observer :** le nom renvoyé par l'API (`name`) n'est pas toujours celui qu'on a demandé, car l'API répond avec la station ou la commune la plus proche. Pour identifier une ville, on utilisera donc **notre** `city_name` comme clé, et on gardera `id` et `name` à titre d'information.

## 6. Première lecture des valeurs par ville

In [ ]:
df["wind_kmh"] = df["wind.speed"] * 3.6
cols = ["main.temp", "main.feels_like", "main.humidity", "main.pressure", "wind_kmh", "clouds.all"]
df.groupby("city_name")[cols].agg(["mean", "min", "max"]).round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, label) in zip(axes, [("main.temp", "Température (°C)"),
                                   ("main.humidity", "Humidité (%)"),
                                   ("wind_kmh", "Vent (km/h)")]):
    for city, g in df.sort_values("observed_at_utc").groupby("city_name"):
        ax.plot(g["observed_at_local"].dt.strftime("%d/%m %Hh%M"), g[col], marker="o", label=city)
    ax.set_title(label)
    ax.tick_params(axis="x", rotation=45)
axes[0].legend()
fig.suptitle("Valeurs par ville et par extraction (heure locale)")
fig.tight_layout()
plt.show()

In [ ]:
df.groupby(["city_name", "weather0.main"]).size().unstack(fill_value=0)

## 7. Visualisations avec seaborn

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Chaque point est une extraction : on voit la dispersion par ville
sns.stripplot(data=df, x="city_name", y="main.temp", hue="source_file", size=8, ax=axes[0])
axes[0].set(title="Température par ville et par extraction", xlabel="", ylabel="°C")
axes[0].legend(title="Extraction", fontsize=8)

# Carte de chaleur : villes en lignes, extractions en colonnes
pivot = df.pivot_table(index="city_name", columns="source_file", values="main.temp")
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="coolwarm", ax=axes[1])
axes[1].set(title="Température (°C)", xlabel="Extraction", ylabel="")
axes[1].tick_params(axis="x", rotation=20)
fig.tight_layout()
plt.show()

## 8. Les mêmes questions en SQL avec DuckDB

DuckDB lit directement les fichiers NDJSON avec du SQL, sans base de données à installer. C'est un bon entraînement avant BigQuery, car la syntaxe est proche : on accède aux champs imbriqués avec un point (`payload.main.temp`).

In [ ]:
import duckdb

con = duckdb.connect()
pattern = str(DATA_DIR / "**" / "*.ndjson")
con.sql(f"CREATE OR REPLACE VIEW weather AS SELECT * FROM read_json_auto('{pattern}', filename = true)")
con.sql("DESCRIBE weather").df()

In [ ]:
# Une ligne par ville et par extraction, avec les champs utiles
con.sql('''
    SELECT
        city_name,
        to_timestamp(payload.dt)              AS observed_at_utc,
        payload.main.temp                     AS temperature_c,
        payload.main.humidity                 AS humidite_pct,
        round(payload.wind.speed * 3.6, 1)    AS vent_kmh,
        payload.weather[1].main               AS temps,      -- en SQL DuckDB, les listes commencent à 1
        regexp_extract(filename, 'run[0-9]+') AS extraction
    FROM weather
    ORDER BY city_name, observed_at_utc
''').df()

In [ ]:
# Contrôle des doublons d'observation, comme en section 5
con.sql('''
    SELECT city_name, payload.id AS city_id, to_timestamp(payload.dt) AS observed_at_utc,
           count(*) AS nb_lignes
    FROM weather
    GROUP BY ALL
    HAVING count(*) > 1
''').df()

In [ ]:
# Résumé par ville
con.sql('''
    SELECT city_name,
           count(*)                                  AS nb_mesures,
           round(avg(payload.main.temp), 1)          AS temp_moy,
           min(payload.main.temp)                    AS temp_min,
           max(payload.main.temp)                    AS temp_max,
           round(avg(payload.main.humidity), 0)      AS humidite_moy
    FROM weather
    GROUP BY city_name
    ORDER BY temp_moy DESC
''').df()

## 9. Conclusions et décisions pour la suite

À compléter après exécution, avec ce que tu observes réellement. Les questions à trancher :

| Question | Ce qu'on regarde | Décision pour `stg_weather` (dbt) |
|---|---|---|
| Quelle est la clé d'une ligne ? | Section 5, doublons `(id, dt)` | Proposition : `city_name` + `observed_at`, en dédoublonnant les observations répétées |
| Quels horodatages garder ? | Section 4 | Proposition : `observed_at` (UTC), `observed_at_local`, `extracted_at` |
| Comment traiter `rain.1h` ou `snow.1h` absents ? | Section 3 | Proposition : remplacer par 0, avec un commentaire dans le modèle |
| Quelles unités exposer ? | Section 1 | Proposition : °C, %, hPa, km/h pour le vent (plus lisible que m/s) |
| Quels tests dbt écrire ? | Section 5 | Proposition : `not_null`, `unique` sur la clé, et des plages acceptées (humidité 0 à 100, etc.) |

**Mes observations :**

- …
- …